# Создание гексагональной сетки для оценки риска оползней

## Большой Сочи: районы и пространственная сетка

Этот ноутбук демонстрирует процесс создания гексагональной сетки для территории Большого Сочи с привязкой к административным районам.

### Содержание:
1. Загрузка границ
2. Создание сетки
3. Привязка к районам
4. Визуализация
5. Сохранение результатов
6. Статистика

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import sys

# Добавляем путь к модулям
sys.path.append('../src/dirty')

from create_hex_grid import create_hex_grid, save_results, print_statistics
from geo_utils import load_and_validate

# Настройки matplotlib
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 12

# Константы
CRS_WGS84 = "EPSG:4326"
CRS_UTM = "EPSG:32637"

# Цвета районов
DISTRICT_COLORS = {
    1: "#3498db",  # Адлерский
    2: "#27ae60",  # Хостинский
    3: "#e67e22",  # Центральный
    4: "#9b59b6",  # Лазаревский
}

DISTRICT_NAMES = {
    1: "Адлерский",
    2: "Хостинский",
    3: "Центральный",
    4: "Лазаревский",
}

## 1. Загрузка границ территории

In [ ]:
# Пути к данным
boundary_path = '../data/external/sochi_boundary.geojson'
districts_path = '../data/external/sochi_districts.geojson'

# Загрузка границ Большого Сочи
boundary_gdf = load_and_validate(boundary_path, CRS_WGS84)
print(f"Границы Большого Сочи:")
print(f"  CRS: {boundary_gdf.crs}")
print(f"  Количество полигонов: {len(boundary_gdf)}")
print(f"  Площадь: {boundary_gdf.to_crs(CRS_UTM).geometry.area.sum() / 1_000_000:.2f} км²")

In [ ]:
# Загрузка границ районов
districts_gdf = load_and_validate(districts_path, CRS_WGS84)
print(f"\nГраницы районов:")
print(f"  CRS: {districts_gdf.crs}")
print(f"  Количество районов: {len(districts_gdf)}")

# Вывод названий районов
for idx, row in districts_gdf.iterrows():
    print(f"  - {row.get('name', 'Unknown')}")

## 2. Создание гексагональной сетки

In [ ]:
# Создание сетки
hex_gdf, hex_gdf_wgs84 = create_hex_grid(
    boundary_path=boundary_path,
    districts_path=districts_path,
    side_length=174  # метров
)

## 3. Привязка к районам

In [ ]:
# Проверка распределения ячеек по районам
print("Распределение ячеек по районам:")
print(hex_gdf['district_name'].value_counts().to_string())

# Проверка уникальности cell_id
print(f"\nУникальность cell_id: {hex_gdf['cell_id'].is_unique}")

# Проверка валидности геометрии
print(f"Валидность геометрии: {hex_gdf.geometry.is_valid.all()}")

In [ ]:
# Проверка площадей ячеек
print(f"Статистика площадей ячеек (м²):")
print(f"  Средняя: {hex_gdf['area_m2'].mean():.2f}")
print(f"  Минимальная: {hex_gdf['area_m2'].min():.2f}")
print(f"  Максимальная: {hex_gdf['area_m2'].max():.2f}")
print(f"  Ожидаемая: 78 744 м²")

## 4. Визуализация

In [ ]:
# Визуализация общей карты
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Рисуем ячейки сетки
hex_gdf_wgs84.plot(
    ax=ax,
    column='color',
    linewidth=0.3,
    edgecolor='gray',
    legend=False
)

# Границы районов
districts_gdf.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=2
)

# Легенда
legend_patches = []
for district_id, color in DISTRICT_COLORS.items():
    patch = mpatches.Patch(color=color, label=DISTRICT_NAMES.get(district_id), alpha=0.8)
    legend_patches.append(patch)

ax.legend(handles=legend_patches, loc='lower right', fontsize=10, title="Районы")

ax.set_title(
    f"Большой Сочи: гексагональная сетка по районам\n"
    f"Всего ячеек: {len(hex_gdf):,}",
    fontsize=16,
    fontweight='bold',
    pad=20
)

ax.set_xlabel("Долгота")
ax.set_ylabel("Широта")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../proj_docs/map_districts_overview.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
# Визуализация отдельного района (например, Адлерский)
district_id = 1  # Адлерский
district_name = DISTRICT_NAMES.get(district_id)

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Фильтруем ячейки района
district_mask = hex_gdf_wgs84['district_id'] == district_id
district_grid = hex_gdf_wgs84[district_mask].copy()

# Соседние районы - приглушённые
other_mask = hex_gdf_wgs84['district_id'] != district_id
other_grid = hex_gdf_wgs84[other_mask].copy()

# Рисуем другие районы серым
other_grid['color'] = '#d5d5d5'
other_grid.plot(ax=ax, column='color', linewidth=0.3, edgecolor='gray', alpha=0.5)

# Рисуем ячейки выбранного района
district_color = DISTRICT_COLORS.get(district_id, '#95a5a6')
district_grid['color'] = district_color
district_grid.plot(ax=ax, column='color', linewidth=0.5, edgecolor='#333333')

# Граница района
district_boundary = districts_gdf[districts_gdf['district_id'] == district_id]
if len(district_boundary) > 0:
    district_boundary.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2.5)

# Статистика
cell_count = len(district_grid)
area_km2 = district_grid['area_km2'].sum()

ax.set_title(
    f"{district_name} район: {cell_count:,} ячеек\n"
    f"Площадь: {area_km2:.2f} км²",
    fontsize=16,
    fontweight='bold',
    pad=20
)

ax.set_xlabel("Долгота")
ax.set_ylabel("Широта")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Сохранение результатов

In [ ]:
# Сохранение в различных форматах
save_results(hex_gdf, hex_gdf_wgs84)

# Проверка сохранённых файлов
print("\nСохранённые файлы:")
for root, dirs, files in os.walk('../data/processed'):
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {filepath}: {size_mb:.2f} МБ")

## 6. Статистика

In [ ]:
# Сводная статистика по районам
print_statistics(hex_gdf)

In [ ]:
# Проверка корректности сетки
print("ПРОВЕРКА КОРРЕКТНОСТИ СЕТКИ")
print("="*60)

# 1. Уникальность cell_id
unique_ids = hex_gdf['cell_id'].is_unique
print(f"1. Все cell_id уникальны: {'✓' if unique_ids else '✗'}")

# 2. Валидность геометрии
valid_geom = hex_gdf.geometry.is_valid.all()
print(f"2. Валидность геометрии: {'✓' if valid_geom else '✗'}")

# 3. Площадь ячеек (±1%)
expected_area = 78744
area_ok = ((hex_gdf['area_m2'] - expected_area).abs() / expected_area < 0.01).all()
print(f"3. Площадь ячеек ≈ 78 744 м² (±1%): {'✓' if area_ok else '✗'}")

# 4. district_id есть у всех (или NULL)
has_district = hex_gdf['district_id'].notna().sum()
no_district = hex_gdf['district_id'].isna().sum()
print(f"4. Ячейки с district_id: {has_district}, без: {no_district} {'✓' if has_district > 0 else '✗'}")

# 5. Сумма площадей ≈ площадь Большого Сочи
total_grid_area = hex_gdf['area_km2'].sum()
boundary_area = boundary_gdf.to_crs(CRS_UTM).geometry.area.sum() / 1_000_000
area_diff_pct = abs(total_grid_area - boundary_area) / boundary_area * 100
print(f"5. Сумма площадей ячеек: {total_grid_area:.2f} км²")
print(f"   Площадь границы: {boundary_area:.2f} км²")
print(f"   Разница: {area_diff_pct:.1f}% {'✓' if area_diff_pct < 10 else '⚠'}")

print("="*60)

## 7. Предпросмотр данных

In [ ]:
# Предпросмотр первых 10 ячеек
cols_to_show = [
    'cell_id', 'district_id', 'district_name', 'color',
    'centroid_lon', 'centroid_lat', 'area_m2', 'coverage_pct'
]

print("Предпросмотр данных (первые 10 ячеек):")
display(hex_gdf[cols_to_show].head(10))

---

## Итоги

✅ Создана гексагональная сетка для Большого Сочи  
✅ Ячейки привязаны к четырём районам  
✅ Результаты сохранены в GeoJSON, GeoPackage и CSV  
✅ Созданы визуализации  

### Следующие шаги:
1. Загрузка DEM (цифровой модели рельефа)
2. Извлечение признаков рельефа
3. Загрузка данных об оползнях
4. Создание модели оценки риска